### Bulk Insert

- 대량 데이터를 입력할때 반드시 써야하는 기법

#### 테스트 테이블 생성

```sql
CREATE TABLE emp_bulk (
    empno number primary key,
    ename varchar2(100),
    sal   number
);
```

#### 한 건 씩 INSERT
- 가장 단순한 일반적인 방법. insert 쿼리를 갯수만큼 반복 실행

In [1]:
# import
import oracledb
import time

dsn = 'localhost:1521/XE'
conn = oracledb.connect(user='test', password='test12345', dsn=dsn)
print('Oracle 연결 성공!', conn.version)

cursor = conn.cursor()
sql = """INSERT INTO emp_bulk (empno, ename, sal)
         VALUES (:1, :2, :3) """

start = time.time()  # 트랙잭션 시작시간

for i in range(1, 100001):  # 총 10만건
    cursor.execute(sql, (i, f"직원_{i}", 3000 + (i % 100)))  # dummy 데이터 입력

conn.commit()
end = time.time()  # 트랜잭션 종료시간

print(f'10만건 INSERT 처리시간 : {end - start}')

cursor.close()
conn.close()

Oracle 연결 성공! 21.3.0.0.0
10만건 INSERT 처리시간 : 32.634644985198975


#### 결과분석

- 실행방식 : `execute()` 함수를 10만번 반복실행
- 네트워크 왕복 : 10만번
- 성능 : 무지하게 느림
- 실무에서 몇천만건 ~ 억단위건수에서는 기하급수적으로 느려짐

#### BULK INSERT
- 대량의 데이터에 가장 최적화된 기법
- 아래 테스트 이전 `TRUNCATE TABLE emp_bulk;` 처리 후 실행

In [2]:
# import
import oracledb
import time

dsn = 'localhost:1521/XE'
conn = oracledb.connect(user='test', password='test12345', dsn=dsn)
print('Oracle 연결 성공!', conn.version)

cursor = conn.cursor()
sql = """INSERT INTO emp_bulk (empno, ename, sal)
         VALUES (:1, :2, :3) """

data = []   # INSERT전에 담아둘 리스트
for i in range(1, 100001):  # 총 10만건
    data.append( (i, f"직원_{i}", 3000 + (i % 100)) )  # dummy 데이터 입력

start = time.time()  # 트랙잭션 시작시간

cursor.executemany(sql, data) # 핵심!!!

conn.commit()
end = time.time()  # 트랜잭션 종료시간

print(f'10만건 BULK INSERT 처리시간 : {end - start}')

cursor.close()
conn.close()

Oracle 연결 성공! 21.3.0.0.0
10만건 BULK INSERT 처리시간 : 0.12290525436401367


#### 결과분석

- 실행방식 : 리스트에 담겨있는 전체 데이터를 `executemany()` 한번에 호출로 입력
- 네트워크 왕복 : 1회
- 성능 : 무지하게 빠름
- 실무에서 엑셀, csv 대량 데이터 입력시 주로 사용

#### 진짜 실무 스타일

- 데이터 사이즈가 너무 크면 한번에 리스트가 insert되기 어려움
- 배치방법 : 적당한 사이즈로 나눠서 처리
- 한번 처리보다 조금 느리려도 안전하게 모두 처리됨
- 아래 테스트 이전 `TRUNCATE TABLE emp_bulk;` 처리 후 실행

In [3]:
# import
import oracledb
import time

dsn = 'localhost:1521/XE'
conn = oracledb.connect(user='test', password='test12345', dsn=dsn)
print('Oracle 연결 성공!', conn.version)

cursor = conn.cursor()
sql = """INSERT INTO emp_bulk (empno, ename, sal)
         VALUES (:1, :2, :3) """

batch_size = 1000
data = []   # INSERT전에 담아둘 리스트

start = time.time()  # 트랙잭션 시작시간

for i in range(1, 100001):  # 총 10만건
    data.append( (i, f"직원_{i}", 3000 + (i % 100)) )  # dummy 데이터 입력

    if len(data) == batch_size:
        cursor.executemany(sql, data) # 핵심!!!
        data = []

if data:       
    cursor.executemany(sql, data) # 나머지 남은 데이터 처리

conn.commit()
end = time.time()  # 트랜잭션 종료시간

print(f'10만건 배치 BULK INSERT 처리시간 : {end - start}')

cursor.close()
conn.close()

Oracle 연결 성공! 21.3.0.0.0
10만건 배치 BULK INSERT 처리시간 : 4.402071714401245


#### 실행분석

- 실행방식 : 배치사이즈 만큼 데이터가 모이면 `executemany()` 실행
- 네트워크왕복 : 100번
- 성능 : 일반 배치보다 성능저하. 단, 안정성을 더 고려한 방법
- 한번에 처리될 양이 얼마나 되는지 파악하고 batch_size를 지정
- 정말 데이터가 많을때 고려. 3000만건 정도도 배치필요 없었음